In [2]:
import pandas as pd

**Obtención de servicios DENUE por AGEB**

*Servicios DENUE 2020* 
Clave - Descripción de categoría
* 621610 - Servicios de enfermería a domicilio
* 623311 - Asilos y otras residencias del sector privado para el cuidado de ancianos
* 623312 - Asilos y otras residencias del sector público para el cuidado de ancianos

In [11]:
denue_2023 = pd.read_csv("../../data/raw/2023/denue2023.csv", encoding="latin-1")
denue_2023.info()

/tmp/ipykernel_50514/4106767113.py:1: DtypeWarning: Columns (35) have mixed types. Specify dtype option on import or set low_memory=False.
  denue_2023 = pd.read_csv("../../data/raw/2023/denue2023.csv", encoding="latin-1")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475331 entries, 0 to 475330
Data columns (total 42 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          475331 non-null  int64  
 1   clee        475331 non-null  object 
 2   nom_estab   475329 non-null  object 
 3   raz_social  103642 non-null  object 
 4   codigo_act  475331 non-null  int64  
 5   nombre_act  475331 non-null  object 
 6   per_ocu     475331 non-null  object 
 7   tipo_vial   472103 non-null  object 
 8   nom_vial    475330 non-null  object 
 9   tipo_v_e_1  461193 non-null  object 
 10  nom_v_e_1   467077 non-null  object 
 11  tipo_v_e_2  461694 non-null  object 
 12  nom_v_e_2   467052 non-null  object 
 13  tipo_v_e_3  460714 non-null  object 
 14  nom_v_e_3   467186 non-null  object 
 15  numero_ext  402736 non-null  float64
 16  letra_ext   194797 non-null  object 
 17  edificio    94658 non-null   object 
 18  edificio_e  95201 non-null   object 
 19  nu

In [12]:
denue_2023.sample()

,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
144428,714954,09007461150005451000000000U1,LA 29,NaN,461150,"Comercio al por menor de leche, otros producto...",0 a 5 personas,EJE VIAL,5 SUR (LEYES DE REFORMA),CIRCUITO,...,Iztapalapa,2475,6,5556942100,NaN,NaN,Fijo,19.378971,-99.093383,2014-12


Lo que unicamente utilizaremos sera:
* Código de la clase de actividad
* Entidad federativa
* Municipio 
* Localidad
* Área geoestadística básica

In [13]:
denue_2023 = denue_2023[["codigo_act", "cve_ent", "entidad", "cve_mun", "municipio", "cve_loc", "localidad", "ageb"]]
denue_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475331 entries, 0 to 475330
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   codigo_act  475331 non-null  int64 
 1   cve_ent     475331 non-null  int64 
 2   entidad     475331 non-null  object
 3   cve_mun     475331 non-null  int64 
 4   municipio   475331 non-null  object
 5   cve_loc     475331 non-null  int64 
 6   localidad   475331 non-null  object
 7   ageb        475331 non-null  object
dtypes: int64(4), object(4)
memory usage: 29.0+ MB


Hay localidades no identificadas, pero esperemos a ver si dentro de nuestras actividades siguen habiendo.

In [14]:
denue_2023["cve_ent"] = denue_2023["cve_ent"].astype(str).str.zfill(2)
denue_2023["cve_mun"] = denue_2023["cve_mun"].astype(str).str.zfill(3)
denue_2023["cve_loc"] = denue_2023["cve_loc"].astype(str).str.zfill(4)
denue_2023["ageb"] = denue_2023["ageb"].str.zfill(4)
denue_2023.sample()

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb
193340,461122,09,Ciudad de MÃ©xico,010,Ãlvaro ObregÃ³n,0001,Ãlvaro ObregÃ³n,0578


In [15]:
denue_2023["cvegeo"] = (denue_2023["cve_ent"] + denue_2023["cve_mun"] + denue_2023["cve_loc"] + denue_2023["ageb"])
denue_2023.sample()

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,cvegeo
55823,433110,09,Ciudad de MÃ©xico,002,Azcapotzalco,0001,Azcapotzalco,0877,0900200010877


In [16]:
serv_enf_dom = denue_2023[denue_2023["codigo_act"] == 621610]

In [17]:
serv_enf_dom = serv_enf_dom.groupby("cvegeo", as_index=False)["codigo_act"].count()
serv_enf_dom.rename(columns={"codigo_act":"cantidad_serv_enf_dom"}, inplace=True)
serv_enf_dom

,cvegeo,cantidad_serv_enf_dom
0,0900300011020,1
1,0900600010382,1
2,0900700012174,1
3,0901000011542,1
4,0901100010679,1
5,0901200010163,1
6,0901200011481,1
7,0901400010257,1
8,0901400010276,1
9,0901400010990,1


In [18]:
asilos = denue_2023[denue_2023["codigo_act"].isin([623311, 623312])]
asilos

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,cvegeo
310009,623311,09,Ciudad de MÃ©xico,015,CuauhtÃ©moc,0001,CuauhtÃ©moc,0767,0901500010767
310090,623311,09,Ciudad de MÃ©xico,012,Tlalpan,0001,Tlalpan,0731,0901200010731
310117,623311,09,Ciudad de MÃ©xico,012,Tlalpan,0001,Tlalpan,0958,0901200010958
310120,623312,09,Ciudad de MÃ©xico,014,Benito JuÃ¡rez,0001,Benito JuÃ¡rez,0685,0901400010685
310178,623311,09,Ciudad de MÃ©xico,014,Benito JuÃ¡rez,0001,Benito JuÃ¡rez,0257,0901400010257
...,...,...,...,...,...,...,...,...,...
338515,623311,09,Ciudad de MÃ©xico,013,Xochimilco,0001,Xochimilco,121A,090130001121A
339697,623311,09,Ciudad de MÃ©xico,002,Azcapotzalco,0001,Azcapotzalco,0699,0900200010699
339788,623311,09,Ciudad de MÃ©xico,015,CuauhtÃ©moc,0001,CuauhtÃ©moc,1182,0901500011182
339796,623311,09,Ciudad de MÃ©xico,012,Tlalpan,0001,Tlalpan,038A,090120001038A


In [19]:
asilos = asilos.groupby("cvegeo", as_index=False)["codigo_act"].count()
asilos.rename(columns={"codigo_act":"cantidad_asilos"}, inplace=True)
asilos

,cvegeo,cantidad_asilos
0,0900200010345,2
1,0900200010491,1
2,0900200010699,1
3,0900200010792,1
4,0900200010858,1
...,...,...
89,0901600010694,1
90,0901600010707,1
91,0901600010942,1
92,0901600011156,1


In [20]:
m = serv_enf_dom.merge(asilos, on="cvegeo", how="outer")

In [21]:
m.fillna(0)

,cvegeo,cantidad_serv_enf_dom,cantidad_asilos
0,0900200010345,0.0,2.0
1,0900200010491,0.0,1.0
2,0900200010699,0.0,1.0
3,0900200010792,0.0,1.0
4,0900200010858,0.0,1.0
...,...,...,...
101,0901600010707,0.0,1.0
102,0901600010942,0.0,1.0
103,0901600011156,0.0,1.0
104,0901600011175,0.0,1.0


In [22]:
m.to_csv("../../data/processed/datos_denue_2023.csv")